In [1]:
import datetime
import io
import resource
import time
import urllib.request
from collections import OrderedDict
from pathlib import Path
from Bio.Align import PairwiseAligner
from Bio.PDB import Chain, MMCIFParser, PDBIO, PDBParser
from Bio.PDB.MMCIF2Dict import MMCIF2Dict
from Bio.PDB.Polypeptide import protein_letters_3to1_extended
from Bio.PDB.Residue import Residue

from src.bioutils import SyncedStructure
import pandas as pd

dir(SyncedStructure)

['__class__',
 '__contains__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__firstlineno__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getitem__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__iter__',
 '__le__',
 '__len__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__static_attributes__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_apply_chain_options',
 '_atom_contacts',
 '_build_chains',
 '_ca_records',
 '_chain_entity_map',
 '_compute_interface',
 '_compute_local_alignment',
 '_entity_sequences',
 '_heavy_atoms',
 '_hydrophobicity',
 '_keep_mapped',
 '_local_alignment',
 '_make_ca_trajectory',
 '_make_chain_data',
 '_map_to_reference',
 '_pair_identity',
 '_prepare_dir',
 '_prepare_path',
 '_protein_residues',
 '_reassign',
 '_reference_for_chain',
 '_renumber',
 '_require_chain',
 '_residue_frame',
 '_residue_min_distances',

In [2]:
def _load_fasta(fp):
    with open(fp) as f:
        return "".join(line.strip() for line in f if not line.strip().startswith('>'))

fasta_dir = Path('rawdata/uniprot')        
fastas = fasta_dir.rglob("*.fasta")
refseqs = {f.stem: _load_fasta(f) for f in fastas}
refseqs.keys()

dict_keys(['TIM', 'UHRF1'])

In [3]:
fp = Path('rawdata/candidate_complexes/TIM-DBD_x_UHRF1-FULL_AFM0.cif')
test = SyncedStructure.from_Path(fp)
print(test.chain_ids)
test_renamed = test.rename_chains({'A':'K'})
print(test_renamed.chain_ids)


[PYTHON-INFO] 2026-10-13 21:21:44 | Loaded local structure rawdata/candidate_complexes/TIM-DBD_x_UHRF1-FULL_AFM0.cif | execution time: 0.42s | max memory: 135.4 MB
['B', 'A']
['B', 'K']


In [4]:
fp = Path('rawdata/candidate_complexes/TIM-DBD_x_UHRF1-FULL_AFM0.cif')
test = SyncedStructure.from_Path(fp)
print(test.chain_ids)
test.get_reference_sequence

print("Who is TIM?")
print(test.match_chains(refseqs['TIM']))

print("\nWho is UHRF1?")
print(test.match_chains(refseqs['UHRF1']))

[PYTHON-INFO] 2026-10-13 21:21:44 | Loaded local structure rawdata/candidate_complexes/TIM-DBD_x_UHRF1-FULL_AFM0.cif | execution time: 0.33s | max memory: 142.2 MB
['B', 'A']
Who is TIM?
  chain_id  identity_percent  aligned_residues  query_coverage_percent
0        B             100.0               139                   11.51
1        A             100.0                 5                    0.41

Who is UHRF1?
  chain_id  identity_percent  aligned_residues  query_coverage_percent
0        B             100.0                 5                    0.63
1        A             100.0               793                  100.00


In [5]:
fp = Path('rawdata/candidate_complexes/TIM-DBD_x_UHRF1-FULL_AFM0.cif')
test = SyncedStructure.from_Path(fp)
print(test.chain_ids)

extracted_sequence = test.get_sequence('A')
print(extracted_sequence)
contact_sequence = test.get_contact_sequence('A')
print(contact_sequence)
len(extracted_sequence)==len(contact_sequence)

[PYTHON-INFO] 2026-10-13 21:21:45 | Loaded local structure rawdata/candidate_complexes/TIM-DBD_x_UHRF1-FULL_AFM0.cif | execution time: 0.34s | max memory: 154.4 MB
['B', 'A']
MWIQVRTMDGRQTHTVDSLSRLTKVEELRRKIQELFHVEPGLQRLFYRGKQMEDGHTLFDYEVRLNDTIQLLVRQSLVLPHSTKERDSELSDTDSGCCLGQSESDKSSTHGEAAAETDSRPADEDMWDETELGLYKVNEYVDARDTNMGAWFEAQVVRVTRKAPSRDEPCSSTSRPALEEDVIYHVKYDDYPENGVVQMNSRDVRARARTIIKWQDLEVGQVVMLNYNPDNPKERGFWYDAEISRKRETRTARELYANVVLGDDSLNDCRIIFVDEVFKIERPGEGSPMVDNPMRRKSGPSCKHCKDDVNRLCRVCACHLCGGRQDPDKQLMCDECDMAFHIYCLDPPLSSVPSEDEWYCPECRNDASEVVLAGERLRESKKKAKMASATSSSQRDWGKGMACVGRTKECTIVPSNHYGPIPGIPVGTMWRFRVQVSESGVHRPHVAGIHGRSNDGAYSLVLAGGYEDDVDHGNFFTYTGSGGRDLSGNKRTAEQSCDQKLTNTNRALALNCFAPINDQEGAEAKDWRSGKPVRVVRNVKGGKNSKYAPAEGNRYDGIYKVVKYWPEKGKSGFLVWRYLLRRDDDEPGPWTKEGKDRIKKLGLTMQYPEGYLEALANREREKENSKREEEEQQEGGFASPRTGKGKWKRKSAGGGPSRAGSPRRTSKKTKVEPYSLTAQQSSLIREDKSNAKLWNEVLASLKDRPASGSPFQLFLSKVEETFQCICCQELVFRPITTVCQHNVCKDCLDRSFRAQVFSCPACRYDLGRSYAMQVNQPLQTVLNQLFPGYGNGR
0000000100000000000000000000000

True

In [6]:
from pathlib import Path

out_dir = Path("data/synced_pdbs")
out_dir.mkdir(parents=True, exist_ok=True)

assigned_structures = []
for fp in list(Path("rawdata/candidate_complexes").glob("*.cif")):
    test = SyncedStructure.from_Path(fp)
    if len(test.chain_ids) != 2:
        raise ValueError(f"File {fp.name} does not have exactly 2 chains.")
    tim_df = test.match_chains(refseqs['TIM'])
    uhrf_df = test.match_chains(refseqs['UHRF1'])
    tim_chain = tim_df.sort_values('aligned_residues', ascending=False).iloc[0]['chain_id'] if not tim_df.empty else None
    uhrf_chain = uhrf_df.sort_values('aligned_residues', ascending=False).iloc[0]['chain_id'] if not uhrf_df.empty else None
    if not tim_chain or not uhrf_chain or tim_chain == uhrf_chain or set([tim_chain, uhrf_chain]) != set(test.chain_ids):
        raise ValueError(f"Failed 1:1 assignment for {fp.name}: TIM -> {tim_chain}, UHRF1 -> {uhrf_chain}")
    test.rename_chains({tim_chain: 'A', uhrf_chain: 'B'})
    test.set_reference_sequences({'A': refseqs['TIM'], 'B': refseqs['UHRF1']})
    test.apply_reference_numbering()
    saved = test.save_pdb(out_dir / f"{test.structure_id}.pdb")
    print(f"[PYTHON-INFO] Saved remapped structure: {saved}")
    assigned_structures.append(test)
    print(f"Processed: {fp.name}")

[PYTHON-INFO] 2026-10-13 21:21:46 | Loaded local structure rawdata/candidate_complexes/TIM-NTD_x_UHRF1-FULL_AFM4.cif | execution time: 0.53s | max memory: 195.6 MB
[PYTHON-INFO] Saved remapped structure: data/synced_pdbs/TIM-NTD_x_UHRF1-FULL_AFM4.pdb
Processed: TIM-NTD_x_UHRF1-FULL_AFM4.cif
[PYTHON-INFO] 2026-10-13 21:21:47 | Loaded local structure rawdata/candidate_complexes/TIM-DBD_x_UHRF1-FULL_AFM3.cif | execution time: 0.33s | max memory: 195.6 MB
[PYTHON-INFO] Saved remapped structure: data/synced_pdbs/TIM-DBD_x_UHRF1-FULL_AFM3.pdb
Processed: TIM-DBD_x_UHRF1-FULL_AFM3.cif
[PYTHON-INFO] 2026-10-13 21:21:48 | Loaded local structure rawdata/candidate_complexes/TIM-FULL_x_UHRF1-TTD_AFM3.cif | execution time: 0.63s | max memory: 207.7 MB
[PYTHON-INFO] Saved remapped structure: data/synced_pdbs/TIM-FULL_x_UHRF1-TTD_AFM3.pdb
Processed: TIM-FULL_x_UHRF1-TTD_AFM3.cif
[PYTHON-INFO] 2026-10-13 21:21:49 | Loaded local structure rawdata/candidate_complexes/TIM-NTD_x_UHRF1-PHD_AFM2.cif | execut